In [0]:
#importing necessary library
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sha2, concat_ws

In [0]:
spark = SparkSession.builder.appName("DW").getOrCreate()

In [0]:
# File location and type
base_dir= "/FileStore/tables/"

#loading all tables 
Dim_CustomerType = spark.read\
    .format("csv")\
        .option("header", True)\
            .option("inferSchema",True)\
                .load(base_dir+"Dim_CustomerType.csv")

Dim_ProductCategory = spark.read\
    .format("csv")\
        .option("header", True)\
            .option("inferSchema",True)\
                .load(base_dir+"Dim_ProductCategory.csv")


Dim_StoreRegion = spark.read\
    .format("csv")\
        .option("header", True)\
            .option("inferSchema",True)\
                .load(base_dir+"Dim_StoreRegion.csv")

Fact_Transactions = spark.read\
    .format("csv")\
        .option("header", True)\
            .option("inferSchema",True)\
                .load(base_dir+"Fact_Transactions.csv")


Custom_Mapping_DIM = spark.read\
    .format("csv")\
        .option("header", True)\
            .option("inferSchema",True)\
                .load(base_dir+"Custom_Mapping_DIM.csv")


In [0]:
Dim_CustomerType.display()
Dim_ProductCategory.display()
Dim_StoreRegion.display()

CustomerType,CustomerTypeID,Segment
Retail,1,B2C
Wholesale,2,B2B


ProductCategory,ProductCategoryID,CategoryGroup
Laptop,101,Computing
Mobile,102,Electronics
Tablet,103,Computing


StoreRegion,RegionID,Territory
North,1,A
South,2,B
East,3,C
West,4,D


In [0]:
Fact_Transactions.display()

TransactionID,ProductCategory,StoreRegion,CustomerType,Amount
1,Laptop,North,Retail,1200
2,Mobile,South,Wholesale,800
3,Tablet,East,Retail,300
4,Laptop,West,Retail,1500
5,Tablet,North,Wholesale,400
6,Mobile,East,Retail,900
7,Mobile,South,Retail,850
8,Laptop,West,Wholesale,1700
9,Tablet,East,Retail,350
10,Mobile,North,Retail,950


Now replacing productCategory, StoreRegion and CustomerType with specific primary key form their DIM table

In [0]:
#joining fact and dim table

DIM_FactTxn = Fact_Transactions\
  .join(Dim_ProductCategory, on="ProductCategory", how="left")\
    .join(Dim_CustomerType, on="CustomerType", how="left")\
      .join(Dim_StoreRegion, on="StoreRegion", how="left")

In [0]:
Fact_Txn_final = DIM_FactTxn.select("TransactionID", "ProductCategoryID", "CustomerTypeID", "RegionID", "Amount")

In [0]:
Fact_Txn_final.display()

TransactionID,ProductCategoryID,CustomerTypeID,RegionID,Amount
1,101,1,1,1200
2,102,2,2,800
3,103,1,3,300
4,101,1,4,1500
5,103,2,1,400
6,102,1,3,900
7,102,1,2,850
8,101,2,4,1700
9,103,1,3,350
10,102,1,1,950


In [0]:
Custom_Mapping_DIM.display()

ProductCategory,StoreRegion,CustomerType,MappingLabel
Tablet,East,Retail,Side Head
Laptop,West,Retail,Premium Box
Mobile,North,Wholesale,North Dist Mobile
Tablet,South,Retail,Tablet Push
Laptop,North,Wholesale,Laptop Supply


Initially lets create MappingLabelId for DIM_CustomLabel

In [0]:
DIM_CustomLabel = Custom_Mapping_DIM.withColumn(
    "MappingLabelId",
    sha2(concat_ws("||", 
        col("ProductCategory"), 
        col("StoreRegion"), 
        col("CustomerType"), 
        col("MappingLabel")
    ), 256)
)

In [0]:
DIM_CustomLabel.display()

ProductCategory,StoreRegion,CustomerType,MappingLabel,MappingLabelId
Tablet,East,Retail,Side Head,c799a87abeca2fd57c6289af216b8b08d55339b16291995887c7ec2096ffe5aa
Laptop,West,Retail,Premium Box,eefb8f08341fe3b9efcd42a96738ac9e0dd58c31ec5501fa1dabf59ea7332fe8
Mobile,North,Wholesale,North Dist Mobile,ac8f1f8c85ae88322aa130f77af4659e5844ac9f502edefc2ee13ceed7ae2785
Tablet,South,Retail,Tablet Push,b92788308b6f81c3be09b07920c714bf46d50a7cc2f680cb0fc30f34e52ae855
Laptop,North,Wholesale,Laptop Supply,df2df2543efed13256057a9caf7382d398101568562f3fae4d579a7aa05e90f8


Now joining other Dim Tables and replacing DIM names with IDs

In [0]:
#joining DIM_CustomLabel and dim table

DIM_CustomLabel_joined = DIM_CustomLabel\
  .join(Dim_ProductCategory, on="ProductCategory", how="left")\
    .join(Dim_CustomerType, on="CustomerType", how="left")\
      .join(Dim_StoreRegion, on="StoreRegion", how="left")

In [0]:
DIM_CustomLabel_joined.display()

StoreRegion,CustomerType,ProductCategory,MappingLabel,MappingLabelId,ProductCategoryID,CategoryGroup,CustomerTypeID,Segment,RegionID,Territory
East,Retail,Tablet,Side Head,c799a87abeca2fd57c6289af216b8b08d55339b16291995887c7ec2096ffe5aa,103,Computing,1,B2C,3,C
West,Retail,Laptop,Premium Box,eefb8f08341fe3b9efcd42a96738ac9e0dd58c31ec5501fa1dabf59ea7332fe8,101,Computing,1,B2C,4,D
North,Wholesale,Mobile,North Dist Mobile,ac8f1f8c85ae88322aa130f77af4659e5844ac9f502edefc2ee13ceed7ae2785,102,Electronics,2,B2B,1,A
South,Retail,Tablet,Tablet Push,b92788308b6f81c3be09b07920c714bf46d50a7cc2f680cb0fc30f34e52ae855,103,Computing,1,B2C,2,B
North,Wholesale,Laptop,Laptop Supply,df2df2543efed13256057a9caf7382d398101568562f3fae4d579a7aa05e90f8,101,Computing,2,B2B,1,A


In [0]:
DIM_CustomLabel_Final = DIM_CustomLabel_joined.select(
    "MappingLabelId",
    "ProductCategoryID",
    "RegionID",
    "CustomerTypeID",
    "MappingLabel"
)

In [0]:
DIM_CustomLabel_Final.display()

MappingLabelId,ProductCategoryID,RegionID,CustomerTypeID,MappingLabel
c799a87abeca2fd57c6289af216b8b08d55339b16291995887c7ec2096ffe5aa,103,3,1,Side Head
eefb8f08341fe3b9efcd42a96738ac9e0dd58c31ec5501fa1dabf59ea7332fe8,101,4,1,Premium Box
ac8f1f8c85ae88322aa130f77af4659e5844ac9f502edefc2ee13ceed7ae2785,102,1,2,North Dist Mobile
b92788308b6f81c3be09b07920c714bf46d50a7cc2f680cb0fc30f34e52ae855,103,2,1,Tablet Push
df2df2543efed13256057a9caf7382d398101568562f3fae4d579a7aa05e90f8,101,1,2,Laptop Supply


Now, lets add DIM_CustomLabel's Id to Fact table

In [0]:
Fact_Txn_with_label = Fact_Txn_final\
  .join(
    DIM_CustomLabel_Final,
    [
       "ProductCategoryID",
       "RegionID",
       "CustomerTypeID"
    ],
    how="left"
  )

Creating final Fact_transaction table with custom labelID

In [0]:
Fact_Txn_with_label_final = Fact_Txn_with_label.select("TransactionID", "ProductCategoryID", "CustomerTypeID", "RegionID", "MappingLabelId","Amount")

In [0]:
Fact_Txn_with_label_final.display()

TransactionID,ProductCategoryID,CustomerTypeID,RegionID,MappingLabelId,Amount
1,101,1,1,null,1200
2,102,2,2,null,800
3,103,1,3,c799a87abeca2fd57c6289af216b8b08d55339b16291995887c7ec2096ffe5aa,300
4,101,1,4,eefb8f08341fe3b9efcd42a96738ac9e0dd58c31ec5501fa1dabf59ea7332fe8,1500
5,103,2,1,null,400
6,102,1,3,null,900
7,102,1,2,null,850
8,101,2,4,null,1700
9,103,1,3,c799a87abeca2fd57c6289af216b8b08d55339b16291995887c7ec2096ffe5aa,350
10,102,1,1,null,950


Now Saving All DIM and Fact table in delta file

In [0]:
Dim_CustomerType.write\
  .format("delta")\
    .mode("overwrite")\
      .save("/FileStore/tables/"+ "Dim_CustomerType")

Dim_ProductCategory.write\
  .format("delta")\
    .mode("overwrite")\
      .save("/FileStore/tables/"+ "Dim_ProductCategory")

Dim_StoreRegion.write\
  .format("delta")\
    .mode("overwrite")\
      .save("/FileStore/tables/"+ "Dim_StoreRegion")

DIM_CustomLabel_Final.write\
  .format("delta")\
    .mode("overwrite")\
      .save("/FileStore/tables/"+ "DIM_CustomLabel")

Fact_Txn_with_label_final.write\
  .format("delta")\
    .mode("overwrite")\
      .save("/FileStore/tables/"+ "Fact_Transaction")